# Comparison Notebook
Perbandingan CDSSSD, MDMSMD, dan EAMDSP pada graph dan input yang sama.

Interpretasi ambiguitas pseudocode yang dipakai: destination duplikat diproses
sebagai request terpisah, dan jika `source == destination` maka cost segmen = 0.

In [1]:
from pprint import pprint
from algorithms import (
    dijkstra_shortest_path,
    run_cdsssd,
    run_mdmsmd,
    run_eamdsp,
)


In [2]:
graph = {
    "A": [("B", 4), ("C", 2)],
    "B": [("A", 4), ("C", 1), ("D", 5), ("E", 12)],
    "C": [("A", 2), ("B", 1), ("D", 8), ("E", 10)],
    "D": [("B", 5), ("C", 8), ("E", 2), ("F", 6)],
    "E": [("B", 12), ("C", 10), ("D", 2), ("F", 2)],
    "F": [("D", 6), ("E", 2)],
}

source = "A"
destinations = ["C", "F", "E", "A", "C"]  # sengaja ada source + duplikat


In [3]:
def print_separator(title: str) -> None:
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

print_separator("Helper + Dijkstra Demo")
pprint(dijkstra_shortest_path(graph, source, "F"))



Helper + Dijkstra Demo
{'cost': 12.0, 'path': ['A', 'C', 'B', 'D', 'E', 'F'], 'visited_nodes': 6}


In [4]:
def run_all_algorithms(graph, source, destinations):
    """Jalankan ketiga algoritma pada input yang sama."""
    return {
        'CDSSSD': run_cdsssd(graph, source, destinations),
        'MDMSMD': run_mdmsmd(graph, source, destinations),
        'EAMDSP': run_eamdsp(graph, source, destinations),
    }


In [5]:
print_separator("Demo Eksekusi Semua Algoritma")
results = run_all_algorithms(graph, source, destinations)
for name, data in results.items():
    print(f"\n{name}:")
    pprint(data)



Demo Eksekusi Semua Algoritma

CDSSSD:
{'algorithm': 'CDSSSD',
 'destination_results': [{'cost': 2.0,
                          'destination': 'C',
                          'path': ['A', 'C'],
                          'visited_nodes': 2},
                         {'cost': 12.0,
                          'destination': 'F',
                          'path': ['A', 'C', 'B', 'D', 'E', 'F'],
                          'visited_nodes': 6},
                         {'cost': 10.0,
                          'destination': 'E',
                          'path': ['A', 'C', 'B', 'D', 'E'],
                          'visited_nodes': 5},
                         {'cost': 0.0,
                          'destination': 'A',
                          'path': ['A'],
                          'visited_nodes': 1},
                         {'cost': 2.0,
                          'destination': 'C',
                          'path': ['A', 'C'],
                          'visited_nodes': 2}],
 'source': 'A

In [6]:
print_separator("Ringkasan Perbandingan")
header = f"{'Algorithm':<10} | {'Full Path':<35} | {'Visit Order':<22} | {'Total Cost':<10} | {'Visited':<7}"
print(header)
print('-' * len(header))
for name in ['CDSSSD', 'MDMSMD', 'EAMDSP']:
    data = results[name]
    if name == 'CDSSSD':
        full_path = 'N/A (independent queries)'
    else:
        full_path = ' -> '.join(data['full_path'])
    visit_order = ', '.join(data['visit_order'])
    print(
        f"{name:<10} | {full_path[:35]:<35} | {visit_order[:22]:<22} | "
        f"{data['total_cost']:<10} | {data['total_visited_nodes']:<7}"
    )



Ringkasan Perbandingan
Algorithm  | Full Path                           | Visit Order            | Total Cost | Visited
------------------------------------------------------------------------------------------------
CDSSSD     | N/A (independent queries)           | C, F, E, A, C          | 26.0       | 16     
MDMSMD     | A -> C -> B -> D -> E -> F -> E ->  | C, F, E, A, C          | 26.0       | 18     
EAMDSP     | A -> C -> B -> D -> E -> F          | A, C, C, E, F          | 12.0       | 12     


## Analisis Singkat
- **Perbedaan perilaku**: CDSSSD menghitung query independen, MDMSMD mengikuti urutan input, EAMDSP memilih nearest-next secara iteratif.
- **Perbedaan urutan kunjungan**: CDSSSD dan MDMSMD mengikuti input `destinations`, sedangkan EAMDSP bisa berubah sesuai jarak minimum aktual.
- **Perbedaan total cost**: CDSSSD cenderung tinggi karena restart dari source awal setiap query. MDMSMD bergantung urutan input. EAMDSP sering lebih rendah dari MDMSMD pada kasus tertentu.
- **Perbedaan total visited nodes**: CDSSSD umumnya lebih tinggi karena banyak pencarian dari source yang sama.
- **Kompleksitas waktu kasar**:
  - CDSSSD: `O(k * (E log V))`
  - MDMSMD: `O(k * (E log V))`
  - EAMDSP: `O(k^2 * (E log V))` karena evaluasi semua tujuan tersisa tiap iterasi
  dengan `k = jumlah destination`, `V = jumlah node`, `E = jumlah edge`.